In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import skew
from scipy.special import boxcox1p
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [14]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train = train.drop('Id', axis=1)
test = test.drop('Id', axis=1)

In [15]:
def splittingg(df):
    df_num = df.select_dtypes(include=['float64', 'int64'])
    df_cat = df.select_dtypes(include=['object'])

    return df_num, df_cat

In [16]:
def feature_engineering(df):
    df_feat = df.copy()

    # Additional useful features
    df_feat['TotalSF'] = (
        df_feat['TotalBsmtSF'].fillna(0)
        + df_feat['1stFlrSF'].fillna(0)
        + df_feat['2ndFlrSF'].fillna(0)
    )

    df_feat['TotalPorchSF'] = (
        df_feat['OpenPorchSF'].fillna(0)
        + df_feat['EnclosedPorch'].fillna(0)
        + df_feat['3SsnPorch'].fillna(0)
        + df_feat['ScreenPorch'].fillna(0)
    )

    df_feat['TotalBath'] = (
        df_feat['FullBath'].fillna(0)
        + 0.5 * df_feat['HalfBath'].fillna(0)
        + df_feat['BsmtFullBath'].fillna(0)
        + 0.5 * df_feat['BsmtHalfBath'].fillna(0)
    )

    # House age
    df_feat['HouseAge'] = (
        df_feat['YrSold'] - df_feat['YearBuilt']
    )

    df_feat['RemodAge'] = (
        df_feat['YrSold'] - df_feat['YearRemodAdd']
    )

    df_feat['IsRemod'] = (df_feat['YearRemodAdd'] != df_feat['YearBuilt']).astype(int)

    df_feat['IsNew'] = (df_feat['YrSold'] == df_feat['YearBuilt']).astype(int)

    # Missing value handling / cleanup
    df_feat['LotFrontage'] = df_feat['LotFrontage'].fillna(
        df_feat['LotFrontage'].median()
    )
    df_feat = df_feat.drop('GarageYrBlt', axis=1)
    df_feat['MasVnrArea'] = df_feat['MasVnrArea'].fillna(0)

    return df_feat

In [17]:


def handle_skewness(df_feat, threshold=0.75, lam=0.15):

    df_feat = df_feat.copy()

    # Log-transform the target
    if 'SalePrice' in df_feat.columns:
        df_feat['SalePrice'] = np.log1p(df_feat['SalePrice'])

    # Compute skewness of numeric features
    numeric_feats = df_feat.select_dtypes(include=['float64', 'int64']).columns
    skewed_features = df_feat[numeric_feats].apply(lambda x: skew(x.dropna()))

    # Select features above the skewness threshold
    skewed = skewed_features[abs(skewed_features) > threshold].index
    skewed = skewed.drop('SalePrice', errors='ignore')

    # Apply Box-Cox transform
    for col in skewed:
        df_feat[col] = boxcox1p(df_feat[col], lam)

    return df_feat

In [18]:
def handle_categorical(df_cat):
    df_cat = df_cat.copy()

    # Columns where NaN likely means 'None' (absence of feature)
    none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']
    for col in none_cols:
        if col in df_cat.columns:
            df_cat[col] = df_cat[col].fillna('0')  # Changed from 'None' to '0'

    # Impute remaining categorical missing values with the mode
    for col in df_cat.columns:
        if df_cat[col].isnull().any():
            df_cat[col] = df_cat[col].fillna(df_cat[col].mode()[0])

    # One-hot encode
    df_cat = pd.get_dummies(df_cat, dtype=int)

    return df_cat

In [19]:
def combined(df_feat, df_cat):
    df_final = pd.concat([df_feat, df_cat], axis=1)
    return df_final

In [20]:
# ---- 1. Run the existing pipeline on train and test ----

train_feat = feature_engineering(train)
test_feat  = feature_engineering(test)

train_num, train_cat = splittingg(train_feat)
test_num,  test_cat  = splittingg(test_feat)

# handle_skewness only touches numeric cols (and log1p's SalePrice if present)
train_num = handle_skewness(train_num)
test_num  = handle_skewness(test_num)   # no SalePrice in test, so it's just boxcox on skewed feats

# ---- 2. Encode categoricals TOGETHER so train/test get identical dummy columns ----

n_train = train_cat.shape[0]
cat_combined = pd.concat([train_cat, test_cat], axis=0, ignore_index=True)
cat_combined_encoded = handle_categorical(cat_combined)

train_cat_enc = cat_combined_encoded.iloc[:n_train, :].reset_index(drop=True)
test_cat_enc  = cat_combined_encoded.iloc[n_train:, :].reset_index(drop=True)

# ---- 3. combining ----

train_final = combined(train_num.reset_index(drop=True), train_cat_enc)
test_final  = combined(test_num.reset_index(drop=True), test_cat_enc)



# ---- 4. Split target out of train_final ----

y = train_final['SalePrice']
X = train_final.drop('SalePrice', axis=1)

X_test_final = test_final.copy()
if 'SalePrice' in X_test_final.columns:
    X_test_final = X_test_final.drop('SalePrice', axis=1)

# Safety: make sure test has exactly the same columns as train, in the same order
X_test_final = X_test_final.reindex(columns=X.columns, fill_value=0)

# ---- 4.5. Impute any remaining numeric NaNs (test.csv has a few train.csv doesn't) ----

numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns
train_medians = X[numeric_cols].median()

X[numeric_cols] = X[numeric_cols].fillna(train_medians)
X_test_final[numeric_cols] = X_test_final[numeric_cols].fillna(train_medians)

In [21]:

def scale_features(X_train, X_test=None):
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
    )
    if X_test is not None:
        X_test_scaled = pd.DataFrame(
            scaler.transform(X_test), columns=X_test.columns, index=X_test.index
        )
        return X_train_scaled, X_test_scaled, scaler
    return X_train_scaled, scaler
